# Prepare Data for Blender Render

This notebook does everything needed before opening Blender.
After running all cells you will have `vertex_colors.npy` in `sample_data/`,
which the Blender render scripts read directly.

**Workflow:**
```
phase_colormap.npy  (your data)          isocortex.obj  (from sample_data/)
         │                                        │
         └──────────────┬─────────────────────────┘
                        ▼
              interpolate phase colors
              onto 3D mesh vertices
                        │
                        ▼
             sample_data/vertex_colors.npy
                        │
                        ▼
              → open Blender and run 03_render_blender.py
```

**Prerequisites:**
```
pip install trimesh scipy matplotlib numpy
```

## Configuration

`sample_data/phase_colormap.npy` is already provided in this repo and is used
by default.  If you want to use your own data, replace `PHASE_COLORMAP_PATH`
with the path to your file.

In [ ]:
import os

# Folder containing all sample data (provided in this repo)
SAMPLE_DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "sample_data")

# ── OPTIONAL: replace with your own phase_colormap.npy ──────────────────────
PHASE_COLORMAP_PATH = os.path.join(SAMPLE_DATA_DIR, "phase_colormap.npy")
# ────────────────────────────────────────────────────────────────────────────

MESH_PATH   = os.path.join(SAMPLE_DATA_DIR, "isocortex.obj")
OUTPUT_PATH = os.path.join(SAMPLE_DATA_DIR, "vertex_colors.npy")

print("Phase colormap :", PHASE_COLORMAP_PATH, " exists:", os.path.exists(PHASE_COLORMAP_PATH))
print("Mesh           :", MESH_PATH,            " exists:", os.path.exists(MESH_PATH))
print("Output will be :", OUTPUT_PATH)

## Step 1 — Load the phase colormap

`phase_colormap.npy` is a 2D image of the dorsal cortex in which every pixel
represents a location in the Allen CCF coordinate system:

- **Rows** → AP axis, 0 – 13 200 µm (1 320 pixels × 25 µm)
- **Columns** → ML axis, 0 – 11 400 µm (1 140 pixels × 25 µm)
- **Channels** → RGB color, already mapped through a cyclic colormap

Because the image is registered to the Allen CCF, we can directly look up
the color for any 3D brain vertex using its (AP, ML) position.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

phase_rgb = np.load(PHASE_COLORMAP_PATH)
if phase_rgb.max() > 1.0:          # normalise uint8 → float32 if needed
    phase_rgb = (phase_rgb / 255.0).astype(np.float32)

print(f"Shape : {phase_rgb.shape}   dtype : {phase_rgb.dtype}")
print(f"Range : [{phase_rgb.min():.3f}, {phase_rgb.max():.3f}]")

plt.figure(figsize=(6, 5))
plt.imshow(phase_rgb, origin="lower")
plt.title("Phase colormap (top-down view of dorsal cortex)")
plt.xlabel("ML axis (pixels)")
plt.ylabel("AP axis (pixels)")
plt.tight_layout()
plt.show()

## Step 2 — Load the isocortex mesh

`isocortex.obj` is the 3D surface mesh of the dorsal isocortex, exported from
the Allen CCF 25 µm atlas.  Each vertex has an (AP, DV, ML) position in the
same coordinate space as the phase image above.

We inspect the axis ranges to confirm the coordinate system is as expected.

In [ ]:
import trimesh

mesh     = trimesh.load(MESH_PATH)
vertices = mesh.vertices        # (N, 3) in Allen CCF µm
normals  = mesh.vertex_normals  # (N, 3)

print(f"Vertices : {vertices.shape}")
print(f"Axis 0 (AP) range : {vertices[:,0].min():.0f} – {vertices[:,0].max():.0f} µm")
print(f"Axis 1 (DV) range : {vertices[:,1].min():.0f} – {vertices[:,1].max():.0f} µm")
print(f"Axis 2 (ML) range : {vertices[:,2].min():.0f} – {vertices[:,2].max():.0f} µm")

## Step 3 — Interpolate phase colors onto mesh vertices

For each vertex in the mesh, we look up its color in the phase image by using
its (AP, ML) position as image coordinates.  Bilinear interpolation fills in
values between pixels.

Vertices that fall outside the image (e.g. ventral or medial surface outside
the recorded FOV) are painted neutral gray.

In [ ]:
from scipy.interpolate import RegularGridInterpolator

H, W    = phase_rgb.shape[:2]
ap_axis = np.linspace(0, 13200, H)   # 1320 voxels × 25 µm
ml_axis = np.linspace(0, 11400, W)   # 1140 voxels × 25 µm

def make_interp(channel):
    return RegularGridInterpolator(
        (ap_axis, ml_axis),
        phase_rgb[:, :, channel],
        method="linear",
        bounds_error=False,
        fill_value=np.nan,     # NaN → vertex is outside the image FOV
    )

# Look up each vertex by (AP = axis 0, ML = axis 2)
query = np.column_stack([vertices[:, 0], vertices[:, 2]])
r = make_interp(0)(query)
g = make_interp(1)(query)
b = make_interp(2)(query)

# Assemble RGBA array; paint out-of-FOV vertices gray
GRAY          = [0.15, 0.15, 0.15, 1.0]
invalid       = np.isnan(r)
vertex_colors = np.column_stack([r, g, b, np.ones(len(vertices))])
vertex_colors[invalid] = GRAY
vertex_colors = np.clip(vertex_colors, 0.0, 1.0).astype(np.float32)

print(f"Total vertices   : {len(vertices)}")
print(f"Colored vertices : {(~invalid).sum()}")
print(f"Gray (outside FOV or masked) : {invalid.sum()}")

## Step 4 — Visual check

Project the vertex colors back onto a 2D top-down scatter plot and compare
with the original phase image.  The colored region should match the cortex
outline in the phase image — if it looks shifted or mirrored, the axis
mapping needs to be corrected.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(phase_rgb, origin="lower")
axes[0].set_title("Original phase colormap")
axes[0].set_xlabel("ML (pixels)");  axes[0].set_ylabel("AP (pixels)")

ap = vertices[:, 0]
ml = vertices[:, 2]
axes[1].scatter(ml, ap, c=vertex_colors[:, :3], s=0.05, marker=".")
axes[1].set_aspect("equal")
axes[1].set_title("All vertices (top-down, CCF µm)")
axes[1].set_xlabel("ML (µm)");  axes[1].set_ylabel("AP (µm)")

valid = ~np.all(np.isclose(vertex_colors[:, :3], 0.15), axis=1)
axes[2].scatter(ml[valid], ap[valid], c=vertex_colors[valid, :3], s=0.05, marker=".")
axes[2].set_aspect("equal")
axes[2].set_title("Colored vertices only")
axes[2].set_xlabel("ML (µm)");  axes[2].set_ylabel("AP (µm)")

plt.tight_layout()
check_path = os.path.join(SAMPLE_DATA_DIR, "interpolation_check.png")
plt.savefig(check_path, dpi=150)
plt.show()
print(f"Saved: {check_path}")

## Step 5 — Save vertex colors

Write `vertex_colors.npy` to `sample_data/`.  This is the only file Blender
needs from this notebook.

In [ ]:
np.save(OUTPUT_PATH, vertex_colors)
print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {vertex_colors.shape}  dtype: {vertex_colors.dtype}")

## Done — ready to render in Blender

Your `sample_data/` folder now contains everything the Blender scripts need:

```
sample_data/
├── root.obj           ← whole-brain shell
├── isocortex.obj      ← cortex surface mesh
└── vertex_colors.npy  ← phase colors per vertex  ✓ just generated
```

**Next steps in Blender:**
1. Open Blender → **Scripting** tab
2. Open one of the render scripts:
   - `scripts/03_render_blender.py`  — metallic / iridescent style
   - `scripts/03b_render_matte.py`   — matte / scientific style
3. Set `data_dir` at the top of the script to the absolute path of `sample_data/`
4. Click **▶ Run Script** (or press **Alt+P**)

The render is saved as a PNG in `sample_data/`.